# Picotron 3-cell SFT API

1. Load a model through Picotron (Unsloth when installed and supported; HF otherwise). 2. Prepare ordinary `datasets` records yourself. 3. Train with a TRL-shaped Picotron trainer. See `docs/dataset_format.md` for the expected fields.


## 1. Load model


In [ ]:
from picotron_sft import load_model

model, tokenizer = load_model(
    'HuggingFaceTB/SmolLM2-135M',
    max_seq_length=256,
    load_in_4bit=False,  # Set True only after Kaggle GPU validation.
)


## 2. Prepare a plain dataset

Dataset construction is deliberately yours. The trainer needs a `text` column; apply your own chat template here if your model needs one.


In [ ]:
from datasets import load_dataset

raw = load_dataset('HuggingFaceTB/smoltalk', 'all', split='train', streaming=True).take(1_000)
records = []
for example in raw:
    messages = example['messages']
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    records.append({'text': text})

from datasets import Dataset
train_dataset = Dataset.from_list(records)


## 3. Train

The class delegates to the existing script-first `run_sft()` implementation. It currently supports AdamW, no warmup, no scheduler, and accumulation of one; unsupported values fail loudly instead of changing training semantics.


In [ ]:
from picotron_sft import PicotronSFTConfig, PicotronSFTTrainer

trainer = PicotronSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=PicotronSFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=2,
        max_seq_length=256,
        max_steps=100,
        learning_rate=2e-5,
    ),
)
losses = trainer.train()
print(f'first_loss={losses[0]:.4f} last_loss={losses[-1]:.4f}')
